# Module 1.1 — Tools: Giving the LLM Hands

**Setup recap.** In `01_naked_llm.ipynb` we talked to a raw Ollama model over HTTP. We saw it hallucinate: it confidently produced *wrong* values for the 2D Ising critical temperature and invented physicists' names. The model has no way to check itself. It's a pure next-token sampler.

**Today's fix.** We give the model **tools**: small, deterministic Python functions it can call. The model decides *when* to call them; we decide *what they compute*. The pattern is called **ReAct** (Reason + Act). The model alternates between:

    Thought:  (free-form reasoning in natural language)
    Action:   <tool-name>(<args>)
    Observation: <whatever the tool returned>

…until it emits `Final Answer: ...`.

**Plan.**

* **Part A**: Build a ReAct agent **from scratch**. We'll define a prompt, a parser, a tool dispatcher, and a `while` loop. You will see every token that flows in and out.
* **Part B**: Rebuild the same thing with the **`smolagents`** library (`ToolCallingAgent`). Shorter code, same behaviour, less control. Good trade for production.

By the end of this notebook you will know exactly what "agent framework" means, because you will have written one.


## 1. Setup — reuse our `chat()` primitive

We keep the same Ollama wiring as notebook 01. If you switched models between notebooks, change `MODEL` below.

> **Model choice.** The course targets `qwen2.5:7b` or  `llama3.1:8b`. Part A (from-scratch ReAct) works fine because we control the parser. Part B (smolagents' `ToolCallingAgent`) is stricter about function-call JSON; with `llama3.1:8b` it may take more steps or occasionally fail. If Part B misbehaves, try `qwen2.5:7b` or `ollama pull mistral:7b` and set `MODEL` accordingly.


In [27]:
import json
import math
import re
from typing import Any, Callable

import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL      = "qwen2.5:7b" #"llama3.1:8b"   # <- swap to "qwen3.5:4b" or "mistral:7b" if available


def chat(messages: list[dict],
         model: str = MODEL,
         temperature: float = 0.0,
         url: str = OLLAMA_URL,
         timeout: int = 120,
         stop: list[str] | None = None) -> str:
    """One round-trip to Ollama's /api/chat. Same as notebook 01, plus a `stop` list.

    We use temperature=0.0 by default here because we want *deterministic*
    tool use, not creative writing. We also accept a `stop` list so the
    model halts as soon as it emits an Observation header (we're going to
    fill that in ourselves).
    """
    options: dict[str, Any] = {"temperature": temperature}
    if stop:
        options["stop"] = stop
    payload = {"model": model, "messages": messages,
               "stream": False, "options": options}
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["message"]["content"]


# smoke test
print(chat([{"role": "user", "content": "Reply with just: Hi all"}]))


Hi all


## 2. Two tiny tools

A "tool" is any pure Python function with:

1. a **name** (what the LLM will type),
2. a **signature** (what arguments it accepts), and
3. a **docstring** (what the LLM reads to decide whether to call it).

We build two tools physicists actually need:

* `calculator(expression)`: provides safe arithmetic, including `sqrt`, `log`, `pi`, etc.
* `unit_converter(value, from_unit, to_unit)`: a hand-written dictionary of conversions we care about (eV↔J, K↔eV, solar masses↔kg, ...).

We intentionally do **not** use Python's built-in `eval()`. It would let the model execute arbitrary code. Instead we restrict the expression to a whitelist of AST node types. That pattern is the right mental model for any tool you expose to an LLM: **make it narrower than it looks**.

The python AST is built-in module that can parse a string into a tree structure without executing it. The expression 2/log(1+sqrt(2)) becomes a tree (like: take 2, apply DIVISION, Call(log), add 1 to ...). ast.parse reads the expression without running it, _safe_eval walks the parsed tree and only allows whitelisted math operations, and the _ALLOWED_* dictionaries define what's whitelisted. It's a sandbox for arithmetic.


In [28]:
import ast
import operator as op

# Whitelisted AST nodes. Anything else -> ValueError. 
# Basically, they define exactly which operations and names are permitted:
_ALLOWED_BINOPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.Pow: op.pow, ast.Mod: op.mod,
    ast.FloorDiv: op.floordiv,
}
_ALLOWED_UNARYOPS = {ast.UAdd: op.pos, ast.USub: op.neg}
_ALLOWED_NAMES: dict[str, Any] = {
    "pi": math.pi, "e": math.e,
    "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
    "exp": math.exp, "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "asin": math.asin, "acos": math.acos, "atan": math.atan,
    "abs": abs, "^": math.pow,
}

#  _safe_eval walks the tree node by node, and for each node it checks: "is this node in my whitelist?" 
#  If yes, evaluate it. If no, raise ValueError and refuse. 
def _safe_eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.Name):
        if node.id in _ALLOWED_NAMES:
            val = _ALLOWED_NAMES[node.id]
            return val if callable(val) else float(val)
        raise ValueError(f"unknown name: {node.id!r}")
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        return _ALLOWED_BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        fn = _ALLOWED_NAMES.get(node.func.id)
        if not callable(fn):
            raise ValueError(f"not a callable: {node.func.id!r}")
        if node.keywords:
            raise ValueError("keyword args not allowed")
        return fn(*[_safe_eval(a) for a in node.args])
    raise ValueError(f"disallowed expression: {ast.dump(node)}")


def calculator(expression: str) -> float:
    """Evaluate a math expression. Supports +, -, *, /, **, and the
    functions sqrt, log, log10, exp, sin, cos, tan, abs, plus the
    constants pi and e. Raises ValueError for anything else.

    Examples:
        calculator("2 * pi * sqrt(3)")  -> 10.8828...
        calculator("log(1 + sqrt(2))")  -> 0.8813...
    """
    tree = ast.parse(expression, mode="eval")
    return float(_safe_eval(tree))


# quick checks
print("2*pi*sqrt(3) =", calculator("2*pi*sqrt(3)"))
print("T_c (2D Ising) = 2/log(1+sqrt(2)) =", calculator("2/log(1+sqrt(2))"))
try:
    calculator("__import__('os').system('echo pwned')")
except ValueError as e:
    print("safely rejected:", e)


2*pi*sqrt(3) = 10.882796185405306
T_c (2D Ising) = 2/log(1+sqrt(2)) = 2.269185314213022
safely rejected: disallowed expression: Call(func=Attribute(value=Call(func=Name(id='__import__', ctx=Load()), args=[Constant(value='os')], keywords=[]), attr='system', ctx=Load()), args=[Constant(value='echo pwned')], keywords=[])


In [29]:
# A tiny, explicit unit-conversion table. We go through SI as the hub:
# every entry is "how many SI base units is 1 <unit>?".
_TO_SI: dict[str, tuple[str, float]] = {
    # energy (SI: joule)
    "J":   ("energy", 1.0),
    "eV":  ("energy", 1.602176634e-19),
    "keV": ("energy", 1.602176634e-16),
    "MeV": ("energy", 1.602176634e-13),
    "GeV": ("energy", 1.602176634e-10),
    "erg": ("energy", 1.0e-7),
    # length (SI: metre)
    "m":   ("length", 1.0),
    "km":  ("length", 1.0e3),
    "cm":  ("length", 1.0e-2),
    "mm":  ("length", 1.0e-3),
    "nm":  ("length", 1.0e-9),
    "A":   ("length", 1.0e-10),   # angstrom
    # mass (SI: kilogram)
    "kg":   ("mass", 1.0),
    "g":    ("mass", 1.0e-3),
    "Msun": ("mass", 1.98892e30),
    # temperature is a special case: handled below
}

# constants for energy <-> temperature via k_B
_K_B_JPK = 1.380649e-23   # Boltzmann constant, J/K


def unit_converter(value: float, from_unit: str, to_unit: str) -> float:
    """Convert `value` from `from_unit` to `to_unit`.

    Supported dimensions:
      - energy : J, eV, keV, MeV, GeV, erg
      - length : m, km, cm, mm, nm, A
      - mass   : kg, g, Msun
      - temperature<->energy via k_B: K <-> J, eV, ...

    Examples:
        unit_converter(1.0, "eV", "J")    -> 1.602e-19
        unit_converter(300, "K",  "eV")   -> 0.02585...
        unit_converter(10,  "Msun", "kg") -> 1.988e+31
    """
    # Case 1: pure temperature (K -> K)
    if from_unit == "K" and to_unit == "K":
        return float(value)

    # Case 2: temperature -> energy
    if from_unit == "K" and to_unit in _TO_SI and _TO_SI[to_unit][0] == "energy":
        joules = float(value) * _K_B_JPK
        return joules / _TO_SI[to_unit][1]

    # Case 3: energy -> temperature
    if to_unit == "K" and from_unit in _TO_SI and _TO_SI[from_unit][0] == "energy":
        joules = float(value) * _TO_SI[from_unit][1]
        return joules / _K_B_JPK

    # Case 4: within one dimension (energy/length/mass)
    if from_unit not in _TO_SI:
        raise ValueError(f"unknown from_unit {from_unit!r}")
    if to_unit not in _TO_SI:
        raise ValueError(f"unknown to_unit {to_unit!r}")
    dim_from, f_si = _TO_SI[from_unit]
    dim_to,   t_si = _TO_SI[to_unit]
    if dim_from != dim_to:
        raise ValueError(f"dimension mismatch: {from_unit} ({dim_from}) vs {to_unit} ({dim_to})")
    return float(value) * f_si / t_si


# quick checks
print("1 eV in J =",      unit_converter(1.0, "eV", "J"))
print("300 K in eV =",    unit_converter(300, "K", "eV"))
print("10 Msun in kg =",  unit_converter(10, "Msun", "kg"))


1 eV in J = 1.602176634e-19
300 K in eV = 0.025851999786435535
10 Msun in kg = 1.98892e+31


## 3. The ReAct system prompt

This is the *entire* contract between us and the model. We tell it:

1. the **available tools** (name + signature + one-line description);
2. the **strict output format** it must follow: one `Thought:` line, then either an `Action:` line or a `Final Answer:` line;
3. that after an `Action:` it must **stop** and wait for us to supply the `Observation:`.

The prompt is what makes the LLM behave like an agent. There is no magic: we are just narrowing the distribution of next tokens.


In [30]:
SYSTEM_PROMPT = """You are a careful scientific assistant. You solve problems by reasoning step by step and using tools.

You have access to the following tools:

- calculator(expression: str) -> float
    Evaluate a math expression. Supports +, -, *, /, **, sqrt, log, exp, sin, cos, tan, abs, pi, e.
    Example: calculator("2*pi*sqrt(3)")

- unit_converter(value: float, from_unit: str, to_unit: str) -> float
    Convert between units. Known units:
      energy: J, eV, keV, MeV, GeV, erg
      length: m, km, cm, mm, nm, A
      mass:   kg, g, Msun
      temperature <-> energy via k_B: K <-> (J, eV, keV, ...)
    Example: unit_converter(300, "K", "eV")

ALWAYS reply using exactly this format, one block at a time:

Thought: <one short sentence explaining what you are about to do>
Action: <tool_name>(<arg1>, <arg2>, ...)

After you write an Action, STOP. Do not write Observation yourself. I will append the real Observation and then hand the turn back to you.

When you have the final numeric answer, reply with:

Thought: <one short sentence>
Final Answer: <the answer, with units>

Rules:
- Use only the two tools above. Do not invent new ones.
- Pass numbers literally, not as Python variables. E.g. Action: calculator("2 + 2"), not Action: calculator(x).
- For unit_converter, quote the unit strings. E.g. Action: unit_converter(300, "K", "eV").
- Keep each Thought to one sentence.
"""
print(SYSTEM_PROMPT[:300], "...")


You are a careful scientific assistant. You solve problems by reasoning step by step and using tools.

You have access to the following tools:

- calculator(expression: str) -> float
    Evaluate a math expression. Supports +, -, *, /, **, sqrt, log, exp, sin, cos, tan, abs, pi, e.
    Example: calc ...


## 4. Parsing the model's reply

The model emits something like:

```
Thought: I need to compute 2*pi*sqrt(3).
Action: calculator("2*pi*sqrt(3)")
```

We need to decide, turn by turn, whether the model wants to **call a tool** or **give a final answer**. A pair of regexes is sufficient. But note, we are doing something genuinely fragile: we are *pattern-matching on prose*. That fragility is exactly why frameworks like `smolagents` switch to structured JSON tool-calls instead (Part B).


In [31]:
# Regex for the last "Action: name(args)" line in a message.
_ACTION_RE = re.compile(
    r"Action:\s*([A-Za-z_][A-Za-z0-9_]*)\s*\((.*?)\)\s*$",
    re.DOTALL | re.MULTILINE,
)
# Regex for "Final Answer: ..." (consumes the rest of the message).
_FINAL_RE = re.compile(r"Final Answer:\s*(.+)\Z", re.DOTALL)


def parse_action(text: str) -> tuple[str, list[Any]] | None:
    """Return (tool_name, [arg, ...]) or None if no Action line is present.

    Args inside the parentheses are parsed with ast.literal_eval, so only
    Python literals (numbers, strings, lists, tuples, ...) are accepted.
    """
    # We want the LAST Action line, because the model might rehash earlier ones.
    matches = list(_ACTION_RE.finditer(text))
    if not matches:
        return None
    name, raw_args = matches[-1].group(1), matches[-1].group(2).strip()
    # ast.literal_eval needs a tuple to handle multiple args cleanly.
    if raw_args == "":
        args: list[Any] = []
    else:
        try:
            parsed = ast.literal_eval("(" + raw_args + ",)")  # force tuple
        except (SyntaxError, ValueError) as e:
            raise ValueError(f"could not parse args {raw_args!r}: {e}") from e
        args = list(parsed)
    return name, args


def parse_final_answer(text: str) -> str | None:
    m = _FINAL_RE.search(text)
    return m.group(1).strip() if m else None


# demo
demo = """Thought: I need to compute Onsager's T_c.
Action: calculator("2 / log(1 + sqrt(2))")"""
print(parse_action(demo))
print(parse_final_answer("Thought: done.\nFinal Answer: 2.269 (dimensionless, J/k_B units)"))


('calculator', ['2 / log(1 + sqrt(2))'])
2.269 (dimensionless, J/k_B units)


## 5. Tool dispatcher and the ReAct loop

The **dispatcher** is a dictionary `{name: function}`. The **loop** is:

```
while steps < MAX_STEPS:
    reply = chat(history)
    if "Final Answer:" in reply: return reply
    name, args = parse_action(reply)
    obs = TOOLS[name](*args)
    history.append(assistant=reply)
    history.append(user=f"Observation: {obs}")
```

That's it. That's the whole agent. Everything else in "agent frameworks" is logging, retries, parallel tool calls, and prettier printing.


In [32]:
TOOLS: dict[str, Callable[..., Any]] = {
    "calculator":     calculator,
    "unit_converter": unit_converter,
}

MAX_STEPS = 8


def run_react(user_task: str, verbose: bool = True) -> str:
    """Run the ReAct loop on one user task. Returns the final-answer string."""
    history: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_task},
    ]

    for step in range(1, MAX_STEPS + 1):
        # Stop the model as soon as it produces "Observation:" on its own
        # that means it's starting to hallucinate tool outputs instead of
        # yielding control back to us.
        reply = chat(history, stop=["Observation:"])
        if verbose:
            print(f"\n--- step {step} ---\n{reply.rstrip()}")

        final = parse_final_answer(reply)
        if final is not None:
            return final

        parsed = parse_action(reply)
        if parsed is None:
            # Model wandered off-format. Nudge it back.
            history.append({"role": "assistant", "content": reply})
            history.append({"role": "user",
                            "content": "Please continue using the Thought/Action or Final Answer format."})
            continue

        name, args = parsed
        if name not in TOOLS:
            obs = f"ERROR: unknown tool {name!r}. Available: {list(TOOLS)}"
        else:
            try:
                result = TOOLS[name](*args)
                obs = repr(result)
            except Exception as exc:
                obs = f"ERROR: {type(exc).__name__}: {exc}"

        if verbose:
            print(f"Observation: {obs}")

        history.append({"role": "assistant", "content": reply})
        history.append({"role": "user", "content": f"Observation: {obs}"})

    return "(no final answer within MAX_STEPS)"


## 6. Run it on two real physics tasks

**Task 1** — Schwarzschild radius. $r_s = 2GM/c^2$. With $M = 10\,M_\odot$, how many km is that? We'll force the agent to chain `unit_converter` (Msun → kg) with `calculator` (apply the formula) with `unit_converter` again (m → km).

**Task 2** — Ising at $J = 0.5\ \mathrm{eV}$. Onsager: $T_c = 2J/(k_B \ln(1+\sqrt{2}))$. So the agent needs to compute the dimensionless $2/\ln(1+\sqrt{2}) \approx 2.269$, multiply by $J$, then convert energy → kelvin.

These tasks are small enough that you can check the answer by hand, but they're big enough that the LLM *cannot* do them reliably without tools — go back and ask the naked model from notebook 01 if you want a laugh.


In [ ]:
task1 = (
    "Compute the Schwarzschild radius of a 10 solar-mass black hole. "
    "Use G = 6.6743e-11 m^3 kg^-1 s^-2 and c = 2.99792458e8 m/s. "
    "Give the answer in kilometres."
)
answer1 = run_react(task1, verbose=True)
print("\n=============================")
print("FINAL ANSWER (task 1):", answer1)

# Correct answer: 29.532 km



--- step 1 ---
Thought: First, I need to calculate the Schwarzschild radius using the formula \( R_s = \frac{2GM}{c^2} \).
Action: calculator("2*6.6743e-11*10*1.989e30/(2.99792458e8**2)")
Observation: 29541.26555055405

--- step 2 ---
Thought: Now I need to convert the result from meters to kilometers.
Action: unit_converter(29541.26555055405, "m", "km")
Observation: 29.54126555055405

--- step 3 ---
Thought: The Schwarzschild radius of a 10 solar-mass black hole is now calculated in kilometers.
Final Answer: 29.54 km

FINAL ANSWER (task 1): 29.54 km


In [ ]:
task2 = (
    "For a 2D Ising model with nearest-neighbour coupling J = 0.5 eV, "
    "compute the Onsager critical temperature T_c = 2 J / (k_B ln(1 + sqrt(2))) "
    "and report it in kelvin."
)
answer2 = run_react(task2, verbose=True)
print("\n=============================")
print("FINAL ANSWER (task 2):", answer2)


# Correct answer: 13166 K 



--- step 1 ---
Thought: First, I need to calculate the value of T_c using the given formula.
Action: calculator("2 * 0.5 / (8.617333262145E-5 * log(1 + sqrt(2)))")
Observation: 13166.401049970438

--- step 2 ---
Thought: The result from the calculation needs to be converted into Kelvin.
Final Answer: 13166.4 K

FINAL ANSWER (task 2): 13166.4 K


## 7. Part B — the same agent, in `smolagents`

The loop we just wrote works, but real code bases use a library. `smolagents` is Hugging Face's minimal agent framework; it's the one we'll use for the rest of the SA Part. The pattern is:

1. **decorate** each tool with `@tool` (it reads your docstring to auto-generate the tool schema);
2. **instantiate** a model adapter (`LiteLLMModel` → Ollama);
3. **instantiate** a `ToolCallingAgent` with your tools + model;
4. **call** `agent.run(task)`.

The big difference vs Part A: `ToolCallingAgent` asks the model for **JSON tool calls**, not free-form `Action:` lines. This is more robust on capable models (GPT-4, Claude, Qwen) but can be brittle on smaller Ollama-served Llamas, which sometimes emit empty argument dicts. If you see that happen, re-read Part A. You now have the tools to debug it.


In [35]:
# smolagents wants tools defined in source-readable files, because its
# @tool decorator inspects the function source. Defining them in a notebook
# cell works *only* because IPython stores the cell source in linecache.
from smolagents import ToolCallingAgent, LiteLLMModel, tool


@tool
def calculator_tool(expression: str) -> float:
    """Evaluate a math expression.

    Supports +, -, *, /, **, sqrt, log, exp, sin, cos, tan, abs, pi, e.

    Args:
        expression: a math expression, e.g. "2*pi*sqrt(3)".
    """
    return calculator(expression)


@tool
def unit_converter_tool(value: float, from_unit: str, to_unit: str) -> float:
    """Convert a value between units.

    Supported dimensions: energy (J, eV, keV, MeV, GeV, erg),
    length (m, km, cm, mm, nm, A), mass (kg, g, Msun), and temperature
    <-> energy via k_B (K <-> J, eV, ...).

    Args:
        value: numeric value to convert.
        from_unit: source unit, e.g. "eV".
        to_unit: target unit, e.g. "J".
    """
    return unit_converter(value, from_unit, to_unit)


# LiteLLM's Ollama adapter expects the "ollama_chat/" prefix.
model = LiteLLMModel(
    model_id=f"ollama_chat/{MODEL}",
    api_base="http://localhost:11434",
    api_key="ollama",        # ignored by Ollama but LiteLLM wants *something*
    temperature=0.0,
)

agent = ToolCallingAgent(
    tools=[calculator_tool, unit_converter_tool],
    model=model,
    max_steps=8,
)

print("Tools registered:", [t.name for t in agent.tools.values()])


Tools registered: ['calculator_tool', 'unit_converter_tool', 'final_answer']


In [36]:
# Run the same ising task through smolagents.
# If you see the agent emit empty-arg tool calls repeatedly, that 
# is a small model (llama3.1:8b, qwen2.5:7b) tool-schema quirk mentioned above.
# Swap MODEL at the top for "qwen3.5:4b" or "mistral:7b" and re-run the cells from §1.
result = agent.run(task2)
print("\n=============================")
print("smolagents FINAL ANSWER:", result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ For a 2D Ising model with nearest-neighbour coupling J = 0.5 eV, compute the Onsager critical temperature T_c = │
│ 2 J / (k_B ln(1 + sqrt(2))) and report it in kelvin.                                                            │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'calculator_tool' with arguments: {'expression': '2 * 0.5 / (8.617333262145e-05 * log(1 +         │
│ sqrt(2)))'}                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 13166.401049970438

[Step 1: Duration 20.45 seconds| Input tokens: 1,555 | Output tokens: 54]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 13166.401049970438}                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final answer: 13166.401049970438

[Step 2: Duration 3.69 seconds| Input tokens: 3,272 | Output tokens: 91]


smolagents FINAL ANSWER: 13166.401049970438


## 8. Key takeaway

An agent is a **loop** around a language model plus a **dispatch table** of deterministic tools. The "framework" is just ergonomics.

Three things to internalise:

1. **Tools narrow the hypothesis space.** The bare LLM could answer anything, so it often answers wrong. A calculator is *mathematically correct by construction*; by delegating arithmetic to it, we forbid an entire class of hallucination.
2. **The prompt *is* the API.** Our whole ReAct protocol lives in one string. Change the string, change the behaviour. This is why prompt engineering is engineering: tiny wording changes shift the distribution.
3. **Fragility is structural.** Parsing prose is brittle. That's why structured tool calling (Part B, and every production framework) moves the contract into JSON. We'll rely on it from Module 1.2 onwards.

## 9. Exercises

Edit and re-run. **The third one is the payoff** — it's the seed of Module 1.2.

1. **New tool — `physical_constant(name)`.** Add a third tool that returns hard-coded CODATA values for `"hbar"`, `"k_B"`, `"G"`, `"c"`, `"e"`, `"m_e"`, `"N_A"`. Update the system prompt, and check the agent uses it instead of baking numerical constants into calculator calls.

2. **Failure modes.** Ask the agent something a tool can't help with, e.g. *"Who wrote the 1944 paper on the 2D Ising model?"* What happens? (It will either hallucinate or give up — both are instructive. Next module we add an arXiv search tool.)

3. **Benchmark vs the naked LLM.** Take the 5 questions you tried in Exercise 3 of notebook 01, re-ask them through `run_react`. Tabulate: how many were improved by tools? Which still fail? This is the motivation for RAG in Module 1.2.

---

**Next up — Module 1.2:** We replace the "baked-in constants" weakness with a **retrieval-augmented** tool that reads real arXiv PDFs. Then the agent stops inventing facts and starts citing.
